# Vision-Language Querying: CLIP + Moondream
**NB 25** | Global scenes | Sentinel-2 | Zero-shot semantic search + VQA

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pathlib

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)
from pygeovision.advanced.vlm.clip_geo import CLIPGeo
from pygeovision.advanced.vlm.moondream_geo import MoondreamGeo


In [ ]:
# Study areas with distinct land cover characteristics
SCENES = {
    'dense_urban'    : ((-0.15,51.47,-0.10,51.52),   'planetary_computer'),   # London
    'tropical_forest': ((-62.5,-3.8,-62.2,-3.5),     'planetary_computer'),   # Amazon
    'desert'         : ((55.1,25.0,55.4,25.3),        'planetary_computer'),   # Dubai desert
    'agricultural'   : ((30.8,30.3,31.2,30.7),        'planetary_computer'),   # Nile Delta
}
BANDS    = ['B02','B03','B04','B08']
DATA_DIR = pathlib.Path('./results/nb25')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# Download one representative scene per land cover type
scene_paths = {}
for scene_type, (bbox, provider) in SCENES.items():
    r  = client.search(bbox=bbox, date_range=('2024-04-01','2024-09-30'),
        providers=[provider], cloud_cover_max=20)
    if r:
        sd  = DATA_DIR/scene_type; sd.mkdir(exist_ok=True)
        dl  = client.download([r[0]], output_dir=str(sd), bands=BANDS,
            post_process=['reproject:EPSG:4326','cog'])
        if dl and dl[0].success:
            scene_paths[scene_type] = dl[0].path
            print(f"  {scene_type:<18}: {dl[0].path.name}")


In [ ]:
# Validate and preprocess ALL scenes — identical pipeline for fair comparison
ready_paths = {}
for scene_type, scene in scene_paths.items():
    bbox = SCENES[scene_type][0]
    rep  = client.validator.validate(str(scene), required_bands=len(BANDS))
    out  = DATA_DIR / f'{scene_type}_preprocessed.tif'
    r    = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=bbox,
        normalise='scale_factor', scale_factor=10000.0,
        model_type='foundation', output_path=str(out))
    ready_paths[scene_type] = str(out)
    print(f"  {scene_type:<18}: {r['steps']}  validated={r['report'] and r['report'].passed}")


In [ ]:
# RemoteCLIP zero-shot classification (RS5M — 5M remote sensing image-text pairs)
clip = CLIPGeo(model_name='remoteclip-l14')

QUERIES = [
    'dense urban area with buildings and roads',
    'tropical rainforest with dense green vegetation',
    'desert landscape with sand dunes and arid terrain',
    'agricultural fields with cropland and irrigation',
    'coastal wetland with water and marshland',
    'industrial zone with warehouses',
]

print("\nRemoteCLIP Zero-Shot Classification Results:")
print(f"{'Scene type':<20}" + "".join(f"{q[:15]:>18}" for q in QUERIES[:3]) + "...")
print('-'*80)

for scene_type, path in ready_paths.items():
    if pathlib.Path(path).exists():
        probs = clip.classify(path, QUERIES)
        pred_idx = probs.argmax()
        print(f"{scene_type:<20}: best match → '{QUERIES[pred_idx][:40]}'  p={probs.max():.2f}")


In [ ]:
# Moondream 2 VQA — natural language scene understanding
moon = MoondreamGeo()

questions = [
    "What is the dominant land use visible in this satellite image?",
    "Are there any signs of water bodies or flooding visible?",
    "Estimate the percentage of vegetation cover.",
    "Describe the infrastructure visible in this scene.",
]

print("\nMoondream 2 VQA on preprocessed satellite scenes:")
for scene_type, path in list(ready_paths.items())[:2]:   # first 2 to limit API calls
    if pathlib.Path(path).exists():
        print(f"\n  Scene: {scene_type}")
        caption = moon.caption(path)
        print(f"  Caption: {caption}")
        for q in questions[:2]:
            answer = moon.vqa(path, q)
            print(f"  Q: {q}")
            print(f"  A: {answer}")


In [ ]:
# Build FAISS index over all preprocessed scenes
if len(ready_paths) > 1:
    paths_list = [p for p in ready_paths.values() if pathlib.Path(p).exists()]
    clip.build_index(str(DATA_DIR), str(DATA_DIR/'satellite_index.faiss'))
    print(f"FAISS index built: {len(paths_list)} scenes indexed")

    # Text-to-scene retrieval
    text_queries = [
        'flooding near agricultural fields',
        'urban sprawl expanding into green areas',
        'forest fire aftermath with burned vegetation',
    ]
    for q in text_queries:
        results_search = clip.search(q, top_k=2)
        print(f"\nQuery: '{q}'")
        for r in results_search:
            print(f"  Match: {r.get('path','')[:50]}  score={r.get('score',0):.3f}")


In [ ]:
# Summary of VLM capabilities
print("="*60)
print("VLM Capabilities — PyGeoVision v2.0")
print("="*60)
print("  RemoteCLIP  : zero-shot classification (RS5M pre-training)")
print("  Moondream 2 : VQA + captioning for satellite scenes")
print("  FAISS search: text-to-image + image-to-image retrieval")
print("  All models  : work on preprocessed/validated GeoTIFF only")
print(f"  Scenes processed: {len(ready_paths)}")


In [ ]:
# Export processed scenes as COG + generate validation reports
for scene_type, p in ready_paths.items():
    if pathlib.Path(p).exists():
        cog_out = DATA_DIR / f'{scene_type}_cog.tif'
        client.postprocess.to_cog(p, str(cog_out))
        rep = client.validator.validate(p)
        print(f'{scene_type:<18}: COG={cog_out.name}  valid={rep.passed}')
print(f'All {len(ready_paths)} preprocessed scenes exported as COG')


**NB25 — VLM Querying** complete.
- Study area: Multi-scene global
- Sensor: Sentinel-2 4-band
- Model: RemoteCLIP RS5M + Moondream 2
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG